# The math of linear regression: the normal equation

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The problem in matrix language
We have n samples and d features. After adding a column of ones (for the intercept), all predictions are one matrix product:

$$\hat{y}=Xw\qquad L(w)=\lVert Xw-y\rVert^2=\sum_i(\hat{y}_i-y_i)^2$$

## Taking the gradient
With the chain rule (or by expanding $(Xw-y)^\top(Xw-y)$):

$$\nabla L=2X^\top(Xw-y)$$

Setting it to zero gives the **normal equation**:

$$X^\top X\,w=X^\top y\quad\Rightarrow\quad w=(X^\top X)^{-1}X^\top y$$

Because L is convex (a bowl), this is the only minimum and the best answer.

## A complete numeric example
Three points $(1,2)$, $(2,4)$, $(3,5)$:

$$X=\begin{bmatrix}1&1\\1&2\\1&3\end{bmatrix}\qquad y=\begin{bmatrix}2\\4\\5\end{bmatrix}$$

| Step | Calculation | Result |
|---|---|---|
| 1. $X^\top X$ | $\begin{bmatrix}1+1+1&1+2+3\\1+2+3&1+4+9\end{bmatrix}$ | $\begin{bmatrix}3&6\\6&14\end{bmatrix}$ |
| 2. $X^\top y$ | $(2+4+5,\ 2+8+15)$ | $(11,\ 25)$ |
| 3. Determinant | $3\times14-6\times6$ | 6 |
| 4. Inverse | $\dfrac16\begin{bmatrix}14&-6\\-6&3\end{bmatrix}$ | |
| 5. Intercept | $(14\times11-6\times25)/6$ | 0.667 |
| 6. Slope | $(-6\times11+3\times25)/6$ | 1.5 |

So $\hat{y}=0.667+1.5x$.

| $x$ | $y$ | $\hat{y}$ | Residual $e=y-\hat{y}$ |
|---|---|---|---|
| 1 | 2 | 2.167 | −0.167 |
| 2 | 4 | 3.667 | 0.333 |
| 3 | 5 | 5.167 | −0.167 |

Check two important facts: the residuals add to zero, and $\sum e_i x_i=-0.167+0.667-0.5=0$. In other words $X^\top e=0$.

## The geometric meaning: a projection
> [The vector y is projected onto the plane spanned by the columns of X. The error is perpendicular to that plane.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_projection.png)

*The vector y is projected onto the plane spanned by the columns of X. The error is perpendicular to that plane.*

$Xw$ can only lie in the "column space of X". The point of that space closest to y is the **perpendicular projection** of y. Requiring the error to be perpendicular to every column means $X^\top(y-Xw)=0$, which is the normal equation. So "least squares" and "perpendicular projection" are the same thing.

## Normal equation or gradient descent?
| | Normal equation | Gradient descent |
|---|---|---|
| Steps | One calculation | Iterative |
| Cost | About $d^3$ (the inverse) | About $nd$ per step |
| Hyperparameters | None | The learning rate |
| Many features (d > 10000) | Slow or impossible | Fine |
| Very large data | Must fit in memory | Works with mini-batches |
| Non-linear models | Does not apply | Works |

## When $X^\top X$ has no inverse
If features are collinear or $d>n$, $X^\top X$ is singular. Two fixes:
- **Ridge:** $w=(X^\top X+\lambda I)^{-1}X^\top y$. Adding $\lambda$ to the diagonal moves every eigenvalue away from zero.
- **Pseudo-inverse via SVD:** `np.linalg.pinv` or `np.linalg.lstsq`, which is what scikit-learn uses.

## The coefficient of determination $R^2$
$$R^2=1-\frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}=1-\frac{0.167}{4.667}=0.964$$

The model explains 96.4% of the variance of y.



In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

x = np.array([1.0, 2.0, 3.0]); y = np.array([2.0, 4.0, 5.0])
X = np.c_[np.ones(3), x]

print("X^T X ="); print(X.T @ X)
print("X^T y =", X.T @ y)
w = np.linalg.solve(X.T @ X, X.T @ y)            # the normal equation
print("w (intercept, slope) =", w.round(4))

e = y - X @ w
print("residuals:", e.round(4), "  sum:", round(e.sum(), 10), "  X^T e:", (X.T @ e).round(10))
print("R^2 =", round(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2), 4))

sk = LinearRegression().fit(x.reshape(-1, 1), y)
print("scikit-learn:", round(sk.intercept_, 4), sk.coef_.round(4))

# Gradient descent reaches the same answer
w_gd = np.zeros(2)
for _ in range(5000):
    w_gd -= 0.05 * 2 * X.T @ (X @ w_gd - y) / 3
print("gradient descent:", w_gd.round(4))

# Collinear features: X^T X is singular, Ridge and the pseudo-inverse still work
rng = np.random.default_rng(0)
a = rng.normal(size=30)
Xc = np.c_[np.ones(30), a, 2 * a]                 # third column = 2 * second column
yc = 1 + 3 * a + rng.normal(0, 0.1, 30)
print("rank of X^T X:", np.linalg.matrix_rank(Xc.T @ Xc), "of 3")
print("ridge        :", np.linalg.solve(Xc.T @ Xc + 0.01 * np.eye(3), Xc.T @ yc).round(3))
print("pseudo-inverse:", (np.linalg.pinv(Xc) @ yc).round(3))

### Exercise

Solve the normal equation by hand for the points $(0,1)$, $(1,3)$, $(2,4)$, $(3,4)$.

In [ ]:
# Your answer here